# 05 - Train a Supervised-Value Model Offline

This notebook is the simple offline check: can the model fit expert Q*?

It uses the same data, embedder, backbone, and action-value head as
`02_train_offline_dqn.ipynb`, but replaces `DqnObjective` with `SvObjective`.
The head is trained to match `info_q_star` at every step (MSE). There is no
Bellman backup and no target network.

1. Load previously collected `Datastore` streams from the Hub (must include `info_q_star`).
2. Build a `DataLoader` that fills one batch (`batch_size=1`) with whole tasks up to `TOKEN_BUDGET` packed tokens.
3. Assemble a `Model` from an embedder, a backbone, and an action-value head.
4. Train with `SvObjective` and save with `push_model_to_hub`.

`Augmenter` remaps `action` ids and, via `input_vector_field` /
`output_vector_field` on `info_q_star`, reorders the Q vector with the **same**
permutation so it stays aligned with the remapped ids.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import SvObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline-sv"      # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-offline-sv"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
MAX_STEPS_PER_EPISODE = 30                    # max steps per episode (matches collect)
MAX_TASK_EPISODES = 20                        # max episodes per task (matches collect)
TOKEN_BUDGET = 16384                          # packed tokens; whole tasks are added until the next one would not fit
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)

LR_BACKBONE = 1e-5                           # AdamW lr for the backbone
LR_HEADS = 1e-5                              # AdamW lr for the heads


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous runs of one `sample_field` from one or more datastores. These examples pass `sample_field="task_index"`, so **each packed group is one task** (every contiguous row that shares a `task_index`). Pass exactly one of `token_budget` or `samples_budget`, plus `batch_size`. `batch_size` is how many times that budget is filled, and `1` is a single fill. Each fill adds whole tasks until the next task would pass the budget. `token_budget` counts packed tokens (step tokens plus each task's group-start tokens). `samples_budget` counts steps in the task. A task that does not fit is left out, and each kept task gets a unique group id. The first task of a fill is left out too when it alone is over the budget, and that raises. These examples pass `token_budget=TOKEN_BUDGET`.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `info_q_star`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
def when_episode_done_nonzero(ctx):
    return "episode_done" in ctx and ctx["episode_done"] != 0

def when_group_start(ctx):
    return bool(ctx["group_start"])

def when_reward_nonzero(ctx):
    return "reward" in ctx and ctx["reward"] != 0.0

def when_step_index_zero_or_group_start(ctx):
    return (ctx.get("step_index") == 0) | bool(ctx["group_start"])

# Pipeline order: augmenter → tokenizer

group_prefix = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. You have 20 episodes to solve the task. The "
    "grid is permuted, so squares are not in order; action ids may be "
    "remapped.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "Predict when you see a new line. Step format: "
    "action,observation[,r=reward][,d=done][,e=episode].\n"
)

# input_vector_field/output_vector_field share the action permute so Q*[perm[a]] stays aligned.

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "output_field": "group_start",
            "format": group_prefix,
            "when": when_group_start,
        },
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "when": when_reward_nonzero,
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "when": when_episode_done_nonzero,
        },
        {
            "type": "text",
            "input_field": "episode_index",
            "format": ",e={field}",
            "when": when_step_index_zero_or_group_start,
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "info_q_star",
        },
    ],
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))
loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    batch_size=1,
    sample_field="task_index",
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only.
- `RegressionHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the head uses that same value so the pieces connect cleanly.

`Tokenizer` text fields match `02`: comma-separated action / observation, `r=` / `d=` when nonzero, and a const newline readout flagged `head_output: True`.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)


head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
    propagate_gradient=1.0,
)

model = Model(
    backbone=backbone,
    heads={"action_value": head},
    action_source="action_value",
    reasoner=None,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`.

1. `inputs, objective_data, group_id = loader.next_batch()` fills one batch (`batch_size=1`) with whole tasks (episode-0 start through `task_done != 0`) until the next task would pass `TOKEN_BUDGET` packed tokens.
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone, and produces flat per-step head predictions.
3. `SvObjective` regresses `predictions["action_value"]` onto `targets=objective_data["info_q_star"]` (MSE over finite entries).
4. `AdamW` steps the heads at `LR_HEADS` and the backbone at `LR_BACKBONE`. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights. There is no `model_polyak` step.

`SvObjective` does not use `episode_done` / `task_done`. Attention stays inside one sample because each `task_index` run is its own group.


In [ ]:
optimizer = AdamW(
    params=[
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": model.heads.parameters(), "lr": LR_HEADS},
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = SvObjective(
    loss_type="mse",
)

def run_train(*, model: Model, optimizer: AdamW, objective: SvObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data, group_id = loader.next_batch()
        objective_data = to_device(data=objective_data, device=device)
        predictions = model(inputs).predictions["action_value"]
        losses, metrics = objective(
            objective_data=objective_data,
            predictions=predictions,
            targets=objective_data["info_q_star"],
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  sv={metrics['value']:.4f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, when-gates, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")